# 1v1 Football DRL — evaluation notebook (Member D)

Plots and tables for the report. Everything is generated from files written by
`train.py` (`logs/<run>/*.csv`) and `test.py` (`results/eval_*.csv`), so the notebook is
reproducible: re-run training/evaluation, then **Run All**.

**Reading guide.** Reward curves show optimisation progress only. The evidence of *learning
football* is match outcomes (W/D/L, goal difference) against the rule-based baseline over
held-out seeds on both attack directions, plus the behaviour diagnostics at the end.

In [ ]:
from pathlib import Path
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:  # plain-python fallback
    display = print

from evaluation.metrics import summarize, wilson_interval, bootstrap_ci

# ---- CONFIGURE ------------------------------------------------------------
RUN = "exp0"                     # run name used with train.py --run-name
LOG_DIR = Path("logs") / RUN
EVAL_CSV = None                  # e.g. "results/eval_final.csv"; None = newest results/eval_*.csv
MA = 100                         # moving-average window (episodes)
# ---------------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

def _latest_eval():
    files = sorted(f for f in glob.glob("results/eval_*.csv") if not f.endswith("_summary.csv"))
    return files[-1] if files else None

EVAL_CSV = EVAL_CSV or _latest_eval()
episodes = pd.read_csv(LOG_DIR / "episodes.csv") if (LOG_DIR / "episodes.csv").exists() else None
updates = pd.read_csv(LOG_DIR / "updates.csv") if (LOG_DIR / "updates.csv").exists() else None
evals = pd.read_csv(LOG_DIR / "evals.csv") if (LOG_DIR / "evals.csv").exists() else None
matches = pd.read_csv(EVAL_CSV) if EVAL_CSV else None
print("training episodes:", None if episodes is None else len(episodes),
      "| eval file:", EVAL_CSV, "| eval rows:", None if matches is None else len(matches))

## 1. Learning curve — undiscounted total reward per episode

In [ ]:
if episodes is None:
    print(f"No training logs at {LOG_DIR}. Run: uv run python train.py --run-name {RUN}")
else:
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(episodes.global_step, episodes.ep_return, lw=0.6, alpha=0.35, label="episode return (raw)")
    ax.plot(episodes.global_step, episodes.ep_return.rolling(MA, min_periods=1).mean(),
            lw=2, label=f"moving average ({MA} episodes)")
    ax.set_xlabel("environment decisions (10 Hz)"); ax.set_ylabel("undiscounted total reward")
    ax.set_title("Learning curve"); ax.legend(); plt.show()

## 2. Outcome learning curves (training episodes)
Rolling win rate and goal difference. Training episodes use *stochastic* actions and random sides,
so these sit below the deterministic evaluation numbers.

In [ ]:
if episodes is not None:
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
    win = (episodes.result == "win").astype(float)
    axes[0].plot(episodes.global_step, win.rolling(MA, min_periods=10).mean())
    axes[0].set_title(f"win rate (rolling {MA})"); axes[0].set_ylim(0, 1)
    axes[1].plot(episodes.global_step, episodes.goal_diff.rolling(MA, min_periods=10).mean())
    axes[1].axhline(0, color="k", lw=0.8); axes[1].set_title(f"goal difference (rolling {MA})")
    for a in axes: a.set_xlabel("environment decisions")
    plt.tight_layout(); plt.show()

## 3. Reward decomposition
Mean per-episode contribution of each reward component (blocks of episodes). A rising shaped return
driven only by `progress_shaping` or steal/recovery terms — while goal difference stays flat — is a
red flag for reward hacking.

In [ ]:
if episodes is not None:
    comp = [c for c in episodes.columns if c.startswith("r_")]
    block = max(1, len(episodes) // 20)
    grouped = episodes.groupby(episodes.index // block)[comp].mean()
    steps = episodes.groupby(episodes.index // block).global_step.max()
    grouped.index = steps.values
    ax = grouped.plot(kind="area", stacked=False, figsize=(10, 4), alpha=0.55)
    ax.set_xlabel("environment decisions"); ax.set_ylabel("mean reward / episode")
    ax.set_title("Reward components"); ax.legend(ncol=3, fontsize=8); plt.show()

## 4. PPO optimisation diagnostics (these diagnose the optimiser, not game skill)

In [ ]:
if updates is not None:
    keys = ["train_policy_loss", "train_value_loss", "train_entropy", "train_approx_kl",
            "train_clip_fraction", "train_explained_variance", "perf_steps_per_sec", "train_lr"]
    keys = [k for k in keys if k in updates]
    fig, axes = plt.subplots(2, 4, figsize=(15, 6))
    for ax, k in zip(axes.ravel(), keys):
        ax.plot(updates.global_step, updates[k]); ax.set_title(k, fontsize=9)
    plt.tight_layout(); plt.show()

## 5. Periodic evaluation during training (deterministic policy, held-out seeds)

In [ ]:
if evals is not None and len(evals):
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
    for ax, k in zip(axes, ["points", "win_rate", "goal_diff"]):
        ax.plot(evals.global_step, evals[k], marker="o"); ax.set_title(k)
        ax.set_xlabel("environment decisions")
    plt.tight_layout(); plt.show()
else:
    print("No periodic evaluation logged yet.")

## 6. Agents vs. baseline — headline results
Win/draw/loss with Wilson 95% intervals; goal difference with bootstrap 95% intervals. Matches cycle
over the four (player identity × attack direction) combinations with identical seeds for every agent.

In [ ]:
if matches is None:
    print("No evaluation CSV. Run: uv run python test.py --agents models/<run>/best.pt --opponent rule_based")
else:
    summary = summarize(matches)
    display(summary[["agent", "matches", "wins", "draws", "losses", "win_rate", "win_ci_low", "win_ci_high",
                     "goals_for", "goals_against", "goal_diff", "goal_diff_ci_low", "goal_diff_ci_high"]].round(3))
    agents = list(summary.agent)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    bottom = np.zeros(len(summary))
    for col, color in (("wins", "#2a9d8f"), ("draws", "#e9c46a"), ("losses", "#e76f51")):
        share = (summary[col] / summary.matches).values
        axes[0].bar(agents, share, bottom=bottom, label=col, color=color); bottom += share
    axes[0].set_ylim(0, 1); axes[0].set_title(f"Outcomes vs {matches.opponent.iloc[0]}"); axes[0].legend()
    gd = summary.goal_diff.values
    err = np.vstack([gd - summary.goal_diff_ci_low.values, summary.goal_diff_ci_high.values - gd])
    axes[1].bar(agents, gd, yerr=err, capsize=5, color="#457b9d"); axes[1].axhline(0, color="k", lw=0.8)
    axes[1].set_title("Mean goal difference (95% bootstrap CI)")
    for a in axes: a.tick_params(axis="x", rotation=20)
    plt.tight_layout(); plt.show()

### Both attack directions (symmetry check)

In [ ]:
if matches is not None:
    side = (matches.groupby(["agent", "attack_dir"])
            .agg(matches=("result", "size"), win_rate=("result", lambda s: (s == "win").mean()),
                 goal_diff=("goal_diff", "mean")).round(3).reset_index())
    side["attack_dir"] = side.attack_dir.map({1: "attacks right", -1: "attacks left"})
    display(side)
    print("A large gap between the two directions would indicate side bias (check observation normalisation).")

## 7. Behaviour diagnostics
Outcome numbers alone can hide degenerate play. Definitions are in `evaluation/metrics.py`.

In [ ]:
if matches is not None:
    cols = ["shots", "goals_per_shot", "kicks", "kick_attempts", "steals_won", "loose_recoveries",
            "lost_to_steal", "lost_boundary", "out_of_bounds_total", "poss_self_frac", "poss_opp_frac",
            "attack_half_ball_frac", "ball_x_attack_mean", "stamina_mean", "stamina_end",
            "sprint_frac", "idle_frac", "x_reversal_rate"]
    behaviour = matches.groupby("agent", sort=False)[cols].mean().round(3)
    display(behaviour.T)

    groups = {"Chance creation": ["shots", "kicks", "kick_attempts"],
              "Ball transitions": ["steals_won", "loose_recoveries", "lost_to_steal", "lost_boundary"],
              "Territory & possession": ["poss_self_frac", "poss_opp_frac", "attack_half_ball_frac", "ball_x_attack_mean"],
              "Energy & movement": ["stamina_mean", "sprint_frac", "idle_frac", "x_reversal_rate"]}
    fig, axes = plt.subplots(1, 4, figsize=(17, 3.8))
    for ax, (title, keys) in zip(axes, groups.items()):
        behaviour[keys].plot(kind="bar", ax=ax, width=0.8, legend=True)
        ax.set_title(title); ax.tick_params(axis="x", rotation=20); ax.legend(fontsize=7)
    plt.tight_layout(); plt.show()

### Stamina profile (original contribution)
Mean and end-of-match stamina per agent, and the share of decisions spent sprinting (effort above the
neutral level that drains stamina).

In [ ]:
if matches is not None:
    stamina = matches.groupby("agent", sort=False)[["stamina_mean", "stamina_min", "stamina_end", "sprint_frac"]].mean()
    ax = stamina.plot(kind="bar", figsize=(8, 3.5)); ax.set_ylim(0, 1.05)
    ax.set_title("Stamina profile (fraction of full) and sprint share"); ax.tick_params(axis="x", rotation=20); plt.show()

## 8. Exploit / degenerate-behaviour flags (plan section 5 checklist)
Heuristic thresholds that point to *where to look*; they do not prove or rule out reward hacking.

In [ ]:
if matches is not None:
    g = matches.groupby("agent", sort=False)
    flags = pd.DataFrame({
        "idle_frac > 0.5 (stalling)": g.idle_frac.mean() > 0.5,
        "x_reversal_rate > 0.3 (shuttling)": g.x_reversal_rate.mean() > 0.3,
        "boundary losses/match > 3": g.lost_boundary.mean() > 3,
        "steals/match > 15 (lose-and-recover loop?)": g.steals_won.mean() > 15,
        "kick_attempts > 40/match (kick spam)": g.kick_attempts.mean() > 40,
        "ep_return high but goal_diff <= 0": (g.ep_return.mean() > 0) & (g.goal_diff.mean() <= 0),
    })
    display(flags)